In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!pip -q install tifffile opencv-python-headless scikit-learn pyyaml matplotlib


In [3]:
from pathlib import Path

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/colab/lsa_convtok_vit/validation")

SCRIPTS_DIR = DRIVE_PROJECT_DIR / "scripts"
RUNNER_SCRIPT = SCRIPTS_DIR / "run_external_validation_colab_local.py"
COMPARISON_SCRIPT = SCRIPTS_DIR / "compare_external_pipelines.py"

RAW_ZIP = DRIVE_PROJECT_DIR / "data" / "fungi.zip"
PRIMARY_CHECKPOINT = DRIVE_PROJECT_DIR / "models" / "final_model_lsa_convtok.pth"
PROVENANCE_CHECKPOINT = DRIVE_PROJECT_DIR / "models" / "final_model_checkpoint.pth"
METADATA_JSON = DRIVE_PROJECT_DIR / "configs" / "external_validation_metadata.json"
CONFIG_YAML = DRIVE_PROJECT_DIR / "configs" / "external_validation_config.yaml"

DRIVE_OUTPUT_DIR = DRIVE_PROJECT_DIR / "outputs_external"
LOCAL_WORK_DIR = Path("/content/external_validation_work")

IMAGE_SIZE = 384
BATCH_SIZE = 64
PREPROCESS_WORKERS = 2
LOADER_WORKERS = 2
BOOTSTRAP_RESAMPLES = 10000
SEED = 42

print("Runner:", RUNNER_SCRIPT)
print("Comparison script:", COMPARISON_SCRIPT)
print("Config:", CONFIG_YAML)
print("Output directory:", DRIVE_OUTPUT_DIR)


Runner: /content/drive/MyDrive/colab/lsa_convtok_vit/validation/scripts/run_external_validation_colab_local.py
Comparison script: /content/drive/MyDrive/colab/lsa_convtok_vit/validation/scripts/compare_external_pipelines.py
Config: /content/drive/MyDrive/colab/lsa_convtok_vit/validation/configs/external_validation_config.yaml
Output directory: /content/drive/MyDrive/colab/lsa_convtok_vit/validation/outputs_external


In [4]:
required_paths = {
    "runner": RUNNER_SCRIPT,
    "comparison_script": COMPARISON_SCRIPT,
    "raw_zip": RAW_ZIP,
    "primary_checkpoint": PRIMARY_CHECKPOINT,
    "provenance_checkpoint": PROVENANCE_CHECKPOINT,
    "metadata_json": METADATA_JSON,
    "config_yaml": CONFIG_YAML,
    "drive_output_parent": DRIVE_OUTPUT_DIR.parent,
}

missing = []
for name, path in required_paths.items():
    exists = path.exists()
    print(f"{name:30s} -> {path} | EXISTS: {exists}")
    if not exists:
        missing.append((name, path))

if missing:
    print("Missing required files:")
    for name, path in missing:
        print(f"  - {name}: {path}")
    raise FileNotFoundError("One or more required files are missing. Upload/copy the refactored scripts and config before running the pipeline.")

print("Planned output directory:", DRIVE_OUTPUT_DIR)
print("Planned local scratch directory:", LOCAL_WORK_DIR)
print("Expected final comparison outputs:")
print("  -", DRIVE_OUTPUT_DIR / "statistics" / "multi_pipeline_comparison" / "multi_pipeline_summary.json")
print("  -", DRIVE_OUTPUT_DIR / "statistics" / "multi_pipeline_comparison" / "multi_pipeline_slide_comparison.csv")


runner                         -> /content/drive/MyDrive/colab/lsa_convtok_vit/validation/scripts/run_external_validation_colab_local.py | EXISTS: True
comparison_script              -> /content/drive/MyDrive/colab/lsa_convtok_vit/validation/scripts/compare_external_pipelines.py | EXISTS: True
raw_zip                        -> /content/drive/MyDrive/colab/lsa_convtok_vit/validation/data/fungi.zip | EXISTS: True
primary_checkpoint             -> /content/drive/MyDrive/colab/lsa_convtok_vit/validation/models/final_model_lsa_convtok.pth | EXISTS: True
provenance_checkpoint          -> /content/drive/MyDrive/colab/lsa_convtok_vit/validation/models/final_model_checkpoint.pth | EXISTS: True
metadata_json                  -> /content/drive/MyDrive/colab/lsa_convtok_vit/validation/configs/external_validation_metadata.json | EXISTS: True
config_yaml                    -> /content/drive/MyDrive/colab/lsa_convtok_vit/validation/configs/external_validation_config.yaml | EXISTS: True
drive_output_p

In [7]:
!python {RUNNER_SCRIPT}   --drive-project-dir {DRIVE_PROJECT_DIR}   --raw-zip {RAW_ZIP}   --primary-checkpoint {PRIMARY_CHECKPOINT}   --provenance-checkpoint {PROVENANCE_CHECKPOINT}   --metadata-json {METADATA_JSON}   --config-yaml {CONFIG_YAML}   --drive-output-dir {DRIVE_OUTPUT_DIR}   --local-work-dir {LOCAL_WORK_DIR}   --comparison-script {COMPARISON_SCRIPT}   --image-size {IMAGE_SIZE}   --batch-size {BATCH_SIZE}   --preprocess-workers {PREPROCESS_WORKERS}   --loader-workers {LOADER_WORKERS}   --bootstrap {BOOTSTRAP_RESAMPLES}   --seed {SEED}   --force-local-reset   --force-preprocess   --run-scale-sensitivity   --run-photometric-sensitivity   --run-multi-pipeline-comparison


Copying to local scratch: /content/drive/MyDrive/colab/lsa_convtok_vit/validation/data/fungi.zip -> /content/external_validation_work/inputs/fungi.zip (9387.8 MB)
Copying to local scratch: /content/drive/MyDrive/colab/lsa_convtok_vit/validation/models/final_model_lsa_convtok.pth -> /content/external_validation_work/checkpoints/final_model_lsa_convtok.pth (20.7 MB)
Copying to local scratch: /content/drive/MyDrive/colab/lsa_convtok_vit/validation/models/final_model_checkpoint.pth -> /content/external_validation_work/checkpoints/final_model_checkpoint.pth (62.1 MB)
Copying to local scratch: /content/drive/MyDrive/colab/lsa_convtok_vit/validation/configs/external_validation_metadata.json -> /content/external_validation_work/inputs/external_validation_metadata.json (0.0 MB)
Copying to local scratch: /content/drive/MyDrive/colab/lsa_convtok_vit/validation/configs/external_validation_config.yaml -> /content/external_validation_work/inputs/external_validation_config.yaml (0.0 MB)
External vali

In [8]:
from pathlib import Path
import json

summary_path = DRIVE_OUTPUT_DIR / "statistics" / "multi_pipeline_comparison" / "multi_pipeline_summary.json"
run_metadata_path = DRIVE_OUTPUT_DIR / "colab_local_run_metadata.json"

for path in [summary_path, run_metadata_path]:
    print(path, "EXISTS:", path.exists())

if summary_path.exists():
    with summary_path.open("r", encoding="utf-8") as f:
        summary = json.load(f)
    print("Primary label:", summary.get("primary_label"))
    print("Comparison pipelines:")
    for item in summary.get("comparison_pipelines", []):
        print("  -", item.get("label"))
    print("Interpretation guardrail:")
    print(summary.get("interpretation_guardrail"))


/content/drive/MyDrive/colab/lsa_convtok_vit/validation/outputs_external/statistics/multi_pipeline_comparison/multi_pipeline_summary.json EXISTS: True
/content/drive/MyDrive/colab/lsa_convtok_vit/validation/outputs_external/colab_local_run_metadata.json EXISTS: True
Primary label: locked_pixel_pipeline
Comparison pipelines:
  - scale_matched_physical_pipeline
  - photometric_mean_std_matched_pipeline
Interpretation guardrail:
locked_pixel_pipeline remains the primary external-validation result. All other pipelines summarized here are post hoc sensitivity analyses and must not replace the primary external-validation estimate or be used for model or threshold selection.
